# Python程序设计（项目实战） · 工单P2.1《Pandas数据处理管道》
**第3周 | 2课时(80分钟) | 理实一体**  | 教师参考版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 数据质量报告加“处理日志”：每步操作与行数变化全程留痕；示范讲解 |
| **B 组长** | 标准达成 | 质量≥88%+日志可复现 |
| **C 操作员** | 基础完成 | 管道完整+质量≥85% |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）

## 一、课前资讯（Predict·预测）
> **预习快问**：dropna/fillna/interpolate各适用什么场景？

> **课前检查**：模板仓库已克隆更新；导学单与实施计划已提交平台

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


## 三、任务要求（Make）
**实施步骤**
1. --peek预览+缺失统计
2. to_datetime三格式统一(combine_first)
3. 清洗+类型转换

**验收标准**（工单《P2.1》「验收标准」四行原文）：`及格(D)`＝`管道完整；质量≥85%；PEP 8≥75%` ｜ `良好(C)`＝`质量≥88%；处理日志可复现` ｜ `优秀(B)`＝`质量≥90%；PEP 8≥85%；文档含使用示例` ｜ `超优(A)`＝`优秀+增量处理/自定义验证规则/100万行<5分钟`
> **口径说明（别把两套字母混为一谈）**：上四行是**工单《P2.1》「验收标准」原文**（及格/良好/优秀/超优），与本课课堂分层（D 跟做保底 / C 基础完成 / B 标准达成 / A 进阶拓展）**不同轴**——课堂 C≈工单及格、课堂 B≈工单良好，工单「优秀」档（如 质量≥90%；PEP 8≥85%；文档含使用示例）课堂阶梯不覆盖。工程化项目的 `output/report.txt` 已按工单原文逐行取证：p11/p21/p33/p55 直接以这四行为分层口径，p12/p22/p41/p66 另设「工单验收四档对照」节；本机取不了证的子项逐条点名，不假判。


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
# -*- coding: utf-8 -*-
"""P2.1 Pandas数据清洗管道 —— 教师完成态(内置脏数据生成+质量评分)
运行: /usr/local/bin/python3.11 -m nbconvert --execute 本notebook
     python W03_pipeline.py --peek   # 预览数据
     python W03_pipeline.py           # 执行管道并评分
"""
import random
import sys

random.seed(2026)

### Starter 2：`def make_raw(path="W03_raw.csv", n=1200):`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def make_raw(path="W03_raw.csv", n=1200):
    """生成脏数据(勿改): 缺失/类型错/越界/重复/时间戳混乱"""
    import csv
    import datetime
    fmts = ["%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%Y%m%d%H%M%S"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["device_id", "ts", "temp", "press", "status"])
        for i in range(n):
            t = round(random.uniform(45, 85), 2)
            if i % 37 == 0:
                t = ""
            if i % 53 == 0:
                t = "N/A"
            if i % 89 == 0:
                t = round(random.uniform(125, 150), 1)
            ts = (datetime.datetime(2026, 3, 1) +
                  datetime.timedelta(minutes=i * 30)).strftime(
                fmts[i % 3 == 0 and 0 or (1 if i % 7 == 0 else 2)])
            row = [f"DEV-{i % 4 + 1:02d}", ts, t,
                   round(random.uniform(.3, .6), 2), "RUN"]
            w.writerow(row)
            if i % 211 == 5:                   # 重复行
                w.writerow(row)
    return path

### Starter 3：`def pipeline(path):`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def pipeline(path):
    """TODO-2~7 教师版完成态: 检查→转型→缺失→异常→去重→验证→输出, 每步行数留痕"""
    import pandas as pd
    df = pd.read_csv(path)
    log = []
    n0 = len(df)
    log.append(f"[0加载] {n0} 行, 列={list(df.columns)}")
    log.append(f"[0检查] temp缺失 {df.temp.isna().sum()} + 非法串{(df.temp.astype(str) == 'N/A').sum()}, "
               f"press缺失 {df.press.isna().sum()}, ts缺失 {df.ts.isna().sum()}")

    # ---- TODO-3 类型转换(先转型后补缺: "N/A"转成NaN才能被插值) ----
    df["temp"] = pd.to_numeric(df.temp, errors="coerce")            # 错误值→NaN
    parts = [pd.to_datetime(df.ts, format=f, errors="coerce")
             for f in ("%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%Y%m%d%H%M%S")]
    ts = parts[0]
    for q in parts[1:]:
        ts = ts.combine_first(q)                                     # 三种格式统一
    bad_ts = ts.isna().sum()
    df["ts"] = ts
    df["device_id"] = df.device_id.astype(str).str.strip()
    log.append(f"[3转型] temp→float, ts→datetime(3格式合并), 无法解析时间 {bad_ts} 条")

    # ---- TODO-2 缺失处理: 决策=时序数值列用组内插值(不丢样本), 其余dropna ----
    # 决策矩阵: 整列缺失>30%→drop列 | 时序连续数值→groupby插值 | 类别型→众数填充
    n_before = len(df)
    df["temp"] = df.groupby("device_id", sort=False).temp.transform(
        lambda s: s.interpolate(method="linear", limit_direction="both"))
    df = df.dropna(subset=["temp", "ts", "press"])                   # 仍缺则丢弃
    log.append(f"[2缺失] temp按设备插值+行dropna: {n_before}→{len(df)} 行, "
               f"剩余缺失 {int(df.isna().sum().sum())}")

    # ---- TODO-4 异常检测: IQR(统计) 与 业务阈值>120 两条规则并集剔除, 分别留痕 ----
    q1, q3 = df.temp.quantile([.25, .75])
    iqr = q3 - q1
    m_iqr = (df.temp < q1 - 1.5 * iqr) | (df.temp > q3 + 1.5 * iqr)
    m_biz = (df.temp > 120) | (df.press < 0.1) | (df.press > 1.0)
    m_all = m_iqr | m_biz
    log.append(f"[4异常] IQR[{q1 - 1.5 * iqr:.1f},{q3 + 1.5 * iqr:.1f}]命中 {int(m_iqr.sum())} "
               f"| 业务阈值(temp>120或press越界)命中 {int(m_biz.sum())} | 并集剔除 {int(m_all.sum())}")
    df = df[~m_all].copy()

    # ---- TODO-5 去重: 同设备同时刻视为重复采集 ----
    n_before = len(df)
    df = df.drop_duplicates(subset=["device_id", "ts"], keep="first")
    log.append(f"[5去重] subset=['device_id','ts'] keep=first: {n_before}→{len(df)} 行, 删 {n_before - len(df)}")

    # ---- TODO-6 质量验证: 范围断言 + 时间单调 ----
    ok_temp = df.temp.between(20, 120).all()
    ok_press = df.press.between(0.1, 1.0).all()
    ok_sorted = df.ts.is_monotonic_increasing or df.groupby("device_id").ts.apply(lambda s: s.is_monotonic_increasing).all()
    assert ok_temp and ok_press, "验证失败: temp/press 仍有越界值"
    log.append(f"[6验证] temp∈[20,120]={ok_temp} press∈[0.1,1.0]={ok_press} 时间有序={bool(ok_sorted)} "
               f"dtype ts={df.ts.dtype}")

    # ---- TODO-7 输出: clean.csv + 处理日志(每步行数全程留痕) ----
    df = df.sort_values(["device_id", "ts"]).reset_index(drop=True)
    df.to_csv("clean.csv", index=False)
    log.append(f"[7输出] clean.csv {len(df)} 行 (清洗率 {1 - len(df) / n0:.1%}) + 本日志共 {len(log) + 1} 条")
    with open("clean_log.txt", "w", encoding="utf-8") as fh:
        fh.write("\n".join(log) + "\n")
    df.attrs["rows_in"] = n0
    return df, log

### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - TODO-2 缺失处理：先做 TODO-3 转型把 `"N/A"` 变成 NaN，再按设备分组 `groupby('device_id').temp.transform(interpolate)`；决策写成"整列缺失>30%→丢列 / 时序数值→插值 / 类别→众数"
> - TODO-3 类型转换：`pd.to_numeric(..., errors='coerce')`；时间戳三种格式各转一次再 `combine_first` 合并
> - TODO-4 异常检测：IQR 用 `quantile([.25,.75]) ± 1.5*iqr`，业务阈值 `temp>120`，两条规则**分别计数**再取并集剔除，理由写进日志
> - TODO-5 去重：`drop_duplicates(subset=['device_id','ts'], keep='first')`
> - TODO-6 验证：`.between(20,120).all()` / `.between(0.1,1.0).all()` 断言，外加 `ts` 是否 datetime
> - TODO-7 输出：`to_csv('clean.csv')` + 每步行数追加进 `log` 列表（A 层要求全程留痕）

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：按 TODO-3→2→4→5→6→7 顺序补全，每补一步运行一次看 `log` 行数变化 |
| **C 操作员** | 只看思路提示独立实现；管道完整且质量评分 ≥85 即达标 |
| **B 组长** | 评分 ≥88 且 `clean_log.txt` 可复现每步行数变化 |
| **A 师傅** | 在 log 中补齐"每步操作 + 行数变化 + 决策理由"三要素，并对 3 种缺失处理方案做对比表 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方“验收标准”四行逐条自查并在工单上签字，未做到的档位如实写「未做」——不得只勾 及格(D) 与 优秀(B+)：`及格(D)`＝`管道完整；质量≥85%；PEP 8≥75%` ｜ `良好(C)`＝`质量≥88%；处理日志可复现` ｜ `优秀(B)`＝`质量≥90%；PEP 8≥85%；文档含使用示例` ｜ `超优(A)`＝`优秀+增量处理/自定义验证规则/100万行<5分钟`

### Starter 4：`def score(df):`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def score(df):
    """质量评分(勿改): 完整性40+准确性30+一致性20+时间性10"""
    import pandas as pd
    s = 0
    s += 40 * (1 - df.isna().mean().mean())
    t = pd.to_numeric(df.temp, errors="coerce")
    s += 30 * ((t.between(20, 120)).mean())
    s += 20 * (df.device_id.str.match(r"DEV-\d{2}").mean())
    s += 10 if "ts" in df and pd.api.types.is_datetime64_any_dtype(df.ts) else 0
    return round(s, 1)

### Starter 5：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
if __name__ == "__main__":
    path = make_raw() if "--peek" not in sys.argv else make_raw()
    if "--peek" in sys.argv:
        import pandas as pd
        print(pd.read_csv(path).head(10))
        print("temp异常值预览:\n", pd.read_csv(path).query("temp in ['','N/A']").head())
    else:
        df, log = pipeline(path)
        print("\n".join(log))
        print("质量评分:", score(df), "(及格线85, 优秀90)")

## 四、提交要求与避坑
- 提交: `学号_姓名_P2.1.py` 或本notebook(含输出)
- ⚠ 链式赋值警告→用.loc或先copy()
- ⚠ to_numeric(errors=coerce)把'N/A'变NaN统一处理
- ⚠ 清洗顺序：类型→缺失→异常→去重，别乱序

---

## 教师参考答案（对照 `教师参考/solutions.py` · `p21()`）

> **发放纪律：本附录勿随学生版下发。** 课堂按层放提示（D层演示照做/C层只给思路行/A层直接做）。

| 学生版 TODO | 教师完成态所在 cell | 关键实现 |
|---|---|---|
| TODO-2 缺失处理 | Starter 3 (`pipeline`) | 按设备 `groupby().transform(interpolate)` 插值，再 `dropna` 兜底；决策矩阵写进 log |
| TODO-3 类型转换 | Starter 3 | `pd.to_numeric(errors='coerce')`；3 种时间格式各 `to_datetime` 后 `combine_first` |
| TODO-4 异常检测 | Starter 3 | IQR `±1.5×iqr` 与业务阈值 `temp>120`/`press` 越界，两条规则分别计数后取并集 |
| TODO-5 去重 | Starter 3 | `drop_duplicates(subset=['device_id','ts'], keep='first')` |
| TODO-6 质量验证 | Starter 3 | `between(20,120).all()` / `between(0.1,1.0).all()` 断言 + ts dtype |
| TODO-7 输出 | Starter 3 | `clean.csv` + `clean_log.txt`（每步行数与理由全程留痕） |
| A 层拓展 | 本 notebook 下方 `p21()` | 与 `pipeline()` 等价的精简参考实现 + 断言 |

In [ ]:
def p21():
    import pandas as pd

    df = pd.read_csv("W03_raw.csv")
    log = [f"加载 {len(df)} 行"]
    log.append(f"temp缺失 {df.temp.isna().sum()}")

    df["temp"] = pd.to_numeric(df.temp, errors="coerce")     # 类型错→NaN
    parts = []
    for fmt in ("%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%Y%m%d%H%M%S"):
        parts.append(pd.to_datetime(df.ts, format=fmt, errors="coerce"))
    ts = parts[0]
    for p in parts[1:]:
        ts = ts.combine_first(p)                            # 三格式统一
    df["ts"] = ts

    df["temp"] = df.temp.interpolate()                       # 决策: 时序插值
    q1, q3 = df.temp.quantile([.25, .75])
    iqr = q3 - q1
    biz = df.temp > 120                                     # 业务阈值
    df = df[~((df.temp < q1 - 1.5 * iqr) | (df.temp > q3 + 1.5 * iqr) | biz)]
    df = df.drop_duplicates(subset=["device_id", "ts"])
    df.to_csv("clean.csv", index=False)

    s = (40 * (1 - df.isna().mean().mean())
         + 30 * df.temp.between(20, 120).mean()
         + 20 * df.device_id.str.match(r"DEV-\d{2}").mean()
         + 10)
    print("\n".join(log))
    print(f"清洗后 {len(df)} 行, 质量评分 {round(s, 1)}")

## 运行与验证

上方参考实现可**直接运行**（合成数据、CPU 离线、结果可复现）。
教师可对照学生版逐 cell 讲解：学生填 TODO → 此处给出完成态。

> 多解法对比见同目录《多解法手册.md》与 `教师参考/solutions_multi*.py`。